## 2019


In [37]:
import pandas as pd

In [38]:
df = pd.read_csv("C:/zzz/oil/scrape/use/exchange/2019/ER_CSV_USD_01012019-31122019.csv")
df

,Exchange rate of (DOLLAR (USD))
0,since 01 Jan 2019 to 31 Dec 2019
1,(BAHT / 1 DOLLAR (USD))
2,Period|Buying Rates Sight Bill|Buying Rates Tr...
3,30 Dec 2019|29.8855|29.9767|30.3313
4,27 Dec 2019|29.8980|29.9858|30.3443
...,...
242,08 Jan 2019|31.7470|31.8330|32.1775
243,07 Jan 2019|31.6905|31.7792|32.1224
244,04 Jan 2019|31.8133|31.9008|32.2298
245,03 Jan 2019|31.9638|32.0442|32.3939


In [39]:
import pandas as pd

# อ่านไฟล์ CSV เป็น text
file_path = r"C:\zzz\oil\scrape\use\exchange\2019\ER_CSV_USD_01012019-31122019.csv"
with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# ข้าม 3 แถวแรก (header)
data_lines = lines[3:]

# แยก column ด้วย "|"
rows = [line.strip().split("|") for line in data_lines if line.strip() != ""]

# สร้าง DataFrame
df = pd.DataFrame(rows, columns=["Period", "Buying_Rates_Sight_Bill", "Buying_Rates_Transfer", "Average_Selling_Rates"])

# ลบแถวที่ Average_Selling_Rates ไม่ใช่ตัวเลข
df = df[df["Average_Selling_Rates"].str.replace(".", "", 1).str.isnumeric()]

# แปลง datatype
df["Average_Selling_Rates"] = df["Average_Selling_Rates"].astype(float)
df["Period"] = pd.to_datetime(df["Period"], errors="coerce")

# เลือกเฉพาะ date และ Average Selling Rates
usd_thb_daily = df[["Period", "Average_Selling_Rates"]].rename(
    columns={"Period": "Date", "Average_Selling_Rates": "USD_BATH"}
)

# เรียงตามวันที่
usd_thb_daily = usd_thb_daily.sort_values("Date").reset_index(drop=True)

usd_thb_daily.head()


,Date,USD_BATH
0,2019-01-02,32.5345
1,2019-01-03,32.3939
2,2019-01-04,32.2298
3,2019-01-07,32.1224
4,2019-01-08,32.1775


In [40]:
import pandas as pd

# อ่านไฟล์ CSV เป็น text
file_path = r"C:\zzz\oil\scrape\use\exchange\2019\ER_CSV_USD_01012019-31122019.csv"
with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# ข้าม 3 แถวแรก (header)
data_lines = lines[3:]

# แยก column ด้วย "|"
rows = [line.strip().split("|") for line in data_lines if line.strip() != ""]

# สร้าง DataFrame
df = pd.DataFrame(rows, columns=[
    "Period", 
    "Buying_Rates_Sight_Bill", 
    "Buying_Rates_Transfer", 
    "Average_Selling_Rates"
])

# ลบแถวที่ Buying_Rates_Sight_Bill ไม่ใช่ตัวเลข
df = df[df["Buying_Rates_Sight_Bill"].str.replace(".", "", 1).str.isnumeric()]

# แปลง datatype
df["Buying_Rates_Sight_Bill"] = df["Buying_Rates_Sight_Bill"].astype(float)
df["Period"] = pd.to_datetime(df["Period"], errors="coerce")

# สร้าง dataframe สุดท้าย เก็บเฉพาะ date และ Buying_Rates_Sight_Bill เป็น USD_BATH
usd_thb_daily = df[["Period", "Buying_Rates_Sight_Bill"]].rename(columns={
    "Period": "Date",
    "Buying_Rates_Sight_Bill": "USD_BATH"
})

# เรียงตามวันที่
usd_thb_daily = usd_thb_daily.sort_values("Date").reset_index(drop=True)

usd_thb_daily.head()


,Date,USD_BATH
0,2019-01-02,32.1166
1,2019-01-03,31.9638
2,2019-01-04,31.8133
3,2019-01-07,31.6905
4,2019-01-08,31.7470


In [41]:
# สร้าง full date range ของปี 2019
full_dates = pd.date_range(start="2019-01-01", end="2019-12-31", freq="D")

# สร้าง DataFrame ของ full date range
full_df = pd.DataFrame({"Date": full_dates})

# merge กับ usd_thb_daily เพื่อเติม row ที่หายไป
usd_thb_daily_full = pd.merge(full_df, usd_thb_daily, on="Date", how="left")

# เติมค่า USD_BATH ที่หายไปด้วย forward fill
usd_thb_daily_full["USD_BATH"] = usd_thb_daily_full["USD_BATH"].fillna(method="ffill")

# เรียงตามวันที่อีกครั้ง
usd_thb_daily_full = usd_thb_daily_full.sort_values("Date").reset_index(drop=True)

usd_thb_daily_full.head(15)


C:\Users\student.COE\AppData\Local\Temp\ipykernel_20508\2188320047.py:11: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  usd_thb_daily_full["USD_BATH"] = usd_thb_daily_full["USD_BATH"].fillna(method="ffill")


,Date,USD_BATH
0,2019-01-01,NaN
1,2019-01-02,32.1166
2,2019-01-03,31.9638
3,2019-01-04,31.8133
4,2019-01-05,31.8133
5,2019-01-06,31.8133
6,2019-01-07,31.6905
7,2019-01-08,31.7470
8,2019-01-09,31.7707
9,2019-01-10,31.6709


In [42]:
# import pandas as pd

# # สมมติ usd_thb_daily ของปี 2022
# # มี columns: date, USD_BATH

# # สร้าง full date range ของปี 2022
# full_dates = pd.date_range(start="2022-01-01", end="2022-12-31", freq="D")
# full_df = pd.DataFrame({"date": full_dates})

# # merge กับ usd_thb_daily
# usd_thb_daily_full = pd.merge(full_df, usd_thb_daily, on="date", how="left")

# # --- Step 1: เติมค่า NaN ต้นปีด้วยปีก่อนหน้า ---
# # โหลดปี 2021 (หรือปีล่าสุดก่อนหน้า) เช่น
# file_path_prev = r"C:\zzz\oil\scrape\use\exchange\2021\ER_CSV_USD_01012021-31122021.csv"
# with open(file_path_prev, "r", encoding="utf-8") as f:
#     lines_prev = f.readlines()

# data_lines_prev = lines_prev[3:]
# rows_prev = [line.strip().split("|") for line in data_lines_prev if line.strip() != ""]
# df_prev = pd.DataFrame(rows_prev, columns=[
#     "Period", 
#     "Buying_Rates_Sight_Bill", 
#     "Buying_Rates_Transfer", 
#     "Average_Selling_Rates"
# ])
# df_prev = df_prev[df_prev["Buying_Rates_Sight_Bill"].str.replace(".", "", 1).str.isnumeric()]
# df_prev["Buying_Rates_Sight_Bill"] = df_prev["Buying_Rates_Sight_Bill"].astype(float)
# df_prev["Period"] = pd.to_datetime(df_prev["Period"], errors="coerce")
# usd_thb_prev = df_prev[["Period", "Buying_Rates_Sight_Bill"]].rename(columns={
#     "Period": "date",
#     "Buying_Rates_Sight_Bill": "USD_BATH"
# })

# # เติมค่า NaN ต้นปี 2022 ด้วยค่า same month/day ของปี 2021
# for idx, row in usd_thb_daily_full.iterrows():
#     if pd.isna(row["USD_BATH"]):
#         prev_year_date = row["date"] - pd.DateOffset(years=1)
#         val = usd_thb_prev.loc[(usd_thb_prev["date"].dt.month == prev_year_date.month) &
#                                (usd_thb_prev["date"].dt.day == prev_year_date.day), "USD_BATH"]
#         if not val.empty:
#             usd_thb_daily_full.at[idx, "USD_BATH"] = val.values[0]

# # --- Step 2: forward fill สำหรับวันว่างอื่น ๆ ---
# usd_thb_daily_full["USD_BATH"] = usd_thb_daily_full["USD_BATH"].fillna(method="ffill")

# usd_thb_daily_full.head(15)


In [43]:
# ตรวจสอบจำนวน NaN ในแต่ละ column
usd_thb_daily_full.isna().sum()


Date        0
USD_BATH    1
dtype: int64

In [44]:
# แสดงแถวทั้งหมดที่ USD_BATH เป็น NaN
nan_rows = usd_thb_daily_full[usd_thb_daily_full["USD_BATH"].isna()]

nan_rows


,Date,USD_BATH
0,2019-01-01,NaN


In [45]:
# เติมค่า NaN ด้วย 32.1924
usd_thb_daily_full["USD_BATH"] = usd_thb_daily_full["USD_BATH"].fillna(32.1924)

# ตรวจสอบว่ามี NaN เหลือไหม
usd_thb_daily_full["USD_BATH"].isna().sum()

usd_thb_daily_full


,Date,USD_BATH
0,2019-01-01,32.1924
1,2019-01-02,32.1166
2,2019-01-03,31.9638
3,2019-01-04,31.8133
4,2019-01-05,31.8133
...,...,...
360,2019-12-27,29.8980
361,2019-12-28,29.8980
362,2019-12-29,29.8980
363,2019-12-30,29.8855


In [46]:
# ตั้งชื่อไฟล์ exchange 2019 เก็บไฟล์บน C:\zzz\oil\scrape\use\exchange\data

import os

# สร้างโฟลเดอร์ถ้ายังไม่มี
output_dir = r"C:\zzz\oil\scrape\use\exchange\data"
os.makedirs(output_dir, exist_ok=True)

# กำหนด path ของไฟล์
output_file = os.path.join(output_dir, "exchange2019.csv")

# บันทึกเป็น CSV
usd_thb_daily_full.to_csv(output_file, index=False)

print(f"Exported CSV to: {output_file}")


Exported CSV to: C:\zzz\oil\scrape\use\exchange\data\exchange2019.csv
